# Dental OPG Tooth Detection
## Custom Faster R-CNN + ResNet50 Backbone
### FDI Two-Digit Numbering System | Framework: PyTorch

Run each cell in order. For resume after disconnect, jump to **Cell 9**.

---
## Cell 1 — Install dependencies

In [2]:
# ─────────────────────────────────────────────────────────────
!pip install torch torchvision -q
!pip install pycocotools -q
!pip install matplotlib opencv-python-headless -q
import os, glob, yaml, shutil, zipfile, time, random
import numpy as np
import torch
import torchvision
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device : {DEVICE}")


Mounted at /content/drive
PyTorch      : 2.11.0+cu128
Torchvision  : 0.26.0+cu128
GPU          : Tesla T4
VRAM         : 15.6 GB
Using device : cuda


---
## Cell 2 — Extract dataset from Drive zip

In [3]:
# ─────────────────────────────────────────────────────────────
zip_path = '/content/drive/MyDrive/Panoramic-Dental-Xray-FDI.v1-stretch---rotate.yolov8 (1).zip'

# The original os.walk loop was incorrect as search_root was a file, not a directory.
# If you intend to search within a directory, you would set search_root to that directory
# and adapt the loop accordingly. For now, we directly use the known zip_path.

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"Zip file not found at the specified path: {zip_path}")

print(f"Found zip: {zip_path}")

EXTRACT_TO = '/content/dental_dataset'
if os.path.exists(EXTRACT_TO):
    shutil.rmtree(EXTRACT_TO)
os.makedirs(EXTRACT_TO, exist_ok=True)

print("Extracting...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(EXTRACT_TO)
print("Done.\n")

yaml_files = glob.glob(f"{EXTRACT_TO}/**/data.yaml", recursive=True)
if not yaml_files:
    raise FileNotFoundError("data.yaml not found in zip!")

DATASET_ROOT = os.path.dirname(yaml_files[0])
YAML_PATH    = yaml_files[0]

with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)

FDI_NAMES   = cfg['names']
NUM_CLASSES = cfg['nc'] + 1

print(f"Dataset root : {DATASET_ROOT}")
print(f"FDI classes  : {FDI_NAMES}")
print(f"Total classes (with background): {NUM_CLASSES}")

for split in ['train', 'valid', 'test']:
    imgs = glob.glob(f"{DATASET_ROOT}/{split}/images/*")
    lbls = glob.glob(f"{DATASET_ROOT}/{split}/labels/*")
    print(f"  {split:6s} -> {len(imgs)} images  {len(lbls)} labels")

Found zip: /content/drive/MyDrive/Panoramic-Dental-Xray-FDI.v1-stretch---rotate.yolov8 (1).zip
Extracting...
Done.

Dataset root : /content/dental_dataset
FDI classes  : ['11', '12', '13', '14', '15', '16', '17', '18', '21', '22', '23', '24', '25', '26', '27', '28', '31', '32', '33', '34', '35', '36', '37', '38', '41', '42', '43', '44', '45', '46', '47', '48']
Total classes (with background): 33
  train  -> 3032 images  3032 labels
  valid  -> 315 images  315 labels
  test   -> 156 images  156 labels


---
## Cell 3 — Custom Dataset class (reads YOLO format labels)

In [4]:
# ─────────────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2
class DentalOPGDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, transforms=None):
        self.img_dir    = img_dir
        self.lbl_dir    = lbl_dir
        self.transforms = transforms
        self.img_paths  = []
        for img_path in sorted(glob.glob(f"{img_dir}/*")):
            stem     = Path(img_path).stem
            lbl_path = os.path.join(lbl_dir, stem + '.txt')
            if os.path.exists(lbl_path):
                self.img_paths.append(img_path)
        print(f"  Loaded {len(self.img_paths)} image-label pairs from {img_dir}")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        stem     = Path(img_path).stem
        lbl_path = os.path.join(self.lbl_dir, stem + '.txt')

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        boxes  = []
        labels = []

        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id            = int(parts[0])
                x_c, y_c, bw, bh = map(float, parts[1:5])
                x1 = (x_c - bw / 2) * W
                y1 = (y_c - bh / 2) * H
                x2 = (x_c + bw / 2) * W
                y2 = (y_c + bh / 2) * H
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(W, x2), min(H, y2)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls_id + 1)

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            'boxes'    : boxes,
            'labels'   : labels,
            'image_id' : torch.tensor([idx]),
            'area'     : (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.zeros(0),
            'iscrowd'  : torch.zeros(len(labels), dtype=torch.int64),
        }

        img_tensor = T.ToTensor()(img)
        if self.transforms:
            img_tensor = self.transforms(img_tensor)

        return img_tensor, target


def get_transforms(train=True):
    transforms = []
    if train:
        transforms.append(T.RandomHorizontalFlip(p=0.5))
        transforms.append(T.ColorJitter(brightness=0.3, contrast=0.3))
    return T.Compose(transforms) if transforms else None


def collate_fn(batch):
    return tuple(zip(*batch))


print("Building datasets...")
train_dataset = DentalOPGDataset(
    img_dir    = f"{DATASET_ROOT}/train/images",
    lbl_dir    = f"{DATASET_ROOT}/train/labels",
    transforms = get_transforms(train=True),
)
valid_dataset = DentalOPGDataset(
    img_dir    = f"{DATASET_ROOT}/valid/images",
    lbl_dir    = f"{DATASET_ROOT}/valid/labels",
    transforms = get_transforms(train=False),
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"Train batches : {len(train_loader)}")
print(f"Valid batches : {len(valid_loader)}")


Building datasets...
  Loaded 3032 image-label pairs from /content/dental_dataset/train/images
  Loaded 315 image-label pairs from /content/dental_dataset/valid/images
Train batches : 758
Valid batches : 158


---
## Cell 4 — Build Faster R-CNN + ResNet50 model

In [6]:
# ─────────────────────────────────────────────────────────────
import torchvision
from torchvision.models.detection          import FasterRCNN
from torchvision.models.detection.rpn      import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models                    import resnet50, ResNet50_Weights
def build_model(num_classes):
    backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    backbone = torch.nn.Sequential(*list(backbone.children())[:-2])
    backbone.out_channels = 2048

    # Fix: Adjust AnchorGenerator to work with a single feature map
    anchor_generator = AnchorGenerator(
        sizes        = ((16, 32, 64, 128, 256),), # One tuple for one feature map, with multiple sizes
        aspect_ratios= ((0.5, 1.0, 2.0),),       # One tuple for one feature map, with multiple aspect ratios
    )

    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names = ['0'],
        output_size   = 7,
        sampling_ratio= 2,
    )

    model = FasterRCNN(
        backbone             = backbone,
        num_classes          = num_classes,
        rpn_anchor_generator = anchor_generator,
        box_roi_pool         = roi_pooler,
        min_size             = 640,
        max_size             = 1024,
        rpn_nms_thresh       = 0.7,
        box_nms_thresh       = 0.4,
        box_score_thresh     = 0.4,
        box_detections_per_img = 50,
    )
    return model


model = build_model(NUM_CLASSES)
model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params/1e6:.1f}M")
print(f"Trainable parameters : {trainable_params/1e6:.1f}M")
print(f"Model ready on       : {DEVICE}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 130MB/s]


Total parameters     : 165.4M
Trainable parameters : 165.4M
Model ready on       : cuda


---
## Cell 5 — Training loop

In [ ]:
# ─────────────────────────────────────────────────────────────
import math
SAVE_DIR = '/content/drive/MyDrive/dental_runs/resnet50_frcnn'
os.makedirs(SAVE_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

EPOCHS        = 50
best_val_loss = float('inf')
history       = {'train_loss': [], 'val_loss': []}


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for i, (images, targets) in enumerate(loader):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses    = sum(loss for loss in loss_dict.values())
        if not math.isfinite(losses.item()):
            print(f"  WARNING: Loss is {losses.item()}, skipping batch")
            continue
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += losses.item()
        if (i + 1) % 10 == 0:
            print(f"    Batch {i+1}/{len(loader)}  loss={losses.item():.4f}  "
                  f"cls={loss_dict.get('loss_classifier', torch.tensor(0)).item():.3f}  "
                  f"box={loss_dict.get('loss_box_reg', torch.tensor(0)).item():.3f}")
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.train()
    total_loss = 0
    for images, targets in loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses    = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
    return total_loss / len(loader)


print(f"Starting training for {EPOCHS} epochs...\n")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss   = validate(model, valid_loader, DEVICE)
    scheduler.step(val_loss)
    elapsed = time.time() - t0
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f"Epoch {epoch:3d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  "
          f"lr={optimizer.param_groups[0]['lr']:.2e}  time={elapsed:.0f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss'   : val_loss,
            'fdi_names'  : FDI_NAMES,
            'num_classes': NUM_CLASSES,
        }, f"{SAVE_DIR}/best.pt")
        print(f"  -> Best model saved (val_loss={val_loss:.4f})")

    if epoch % 5 == 0:
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss'   : val_loss,
            'fdi_names'  : FDI_NAMES,
            'num_classes': NUM_CLASSES,
        }, f"{SAVE_DIR}/checkpoint_epoch{epoch}.pt")
        print(f"  -> Checkpoint saved at epoch {epoch}")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

Starting training for 50 epochs...

    Batch 10/758  loss=2.4687  cls=1.361  box=0.660
    Batch 20/758  loss=2.4287  cls=1.331  box=0.739
    Batch 30/758  loss=2.1293  cls=1.179  box=0.739
    Batch 40/758  loss=1.6764  cls=0.956  box=0.584
    Batch 50/758  loss=1.7470  cls=0.975  box=0.643
    Batch 60/758  loss=2.0456  cls=1.116  box=0.732
    Batch 70/758  loss=1.6491  cls=0.929  box=0.564
    Batch 80/758  loss=1.6023  cls=0.895  box=0.608
    Batch 90/758  loss=1.5077  cls=0.833  box=0.559
    Batch 100/758  loss=1.5200  cls=0.800  box=0.596
    Batch 110/758  loss=1.3527  cls=0.724  box=0.532
    Batch 120/758  loss=1.5162  cls=0.839  box=0.565
    Batch 130/758  loss=1.2786  cls=0.743  box=0.437
    Batch 140/758  loss=1.3031  cls=0.679  box=0.520
    Batch 150/758  loss=1.2987  cls=0.708  box=0.472
    Batch 160/758  loss=1.2897  cls=0.657  box=0.524
    Batch 170/758  loss=1.5792  cls=0.841  box=0.564
    Batch 180/758  loss=1.4683  cls=0.834  box=0.541
    Batch 190/758  

KeyboardInterrupt: 

---
## Cell 6 — Plot training curves

In [7]:
# ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train loss', color='steelblue')
plt.plot(history['val_loss'],   label='Val loss',   color='coral')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/loss_curve.png", dpi=150)
plt.show()


NameError: name 'history' is not defined

<Figure size 1000x400 with 0 Axes>

---
## Cell 7 — Load best model & predict on test images

In [8]:
# ─────────────────────────────────────────────────────────────
import matplotlib.patches as patches
def load_model(checkpoint_path, device):
    ckpt        = torch.load(checkpoint_path, map_location=device)
    num_classes = ckpt['num_classes']
    fdi_names   = ckpt['fdi_names']
    m = build_model(num_classes)
    m.load_state_dict(ckpt['model_state'])
    m.to(device)
    m.eval()
    print(f"Loaded model from epoch {ckpt['epoch']}  val_loss={ckpt['val_loss']:.4f}")
    return m, fdi_names


def predict_opg(model, img_path, fdi_names, device, conf_thresh=0.4):
    img_bgr    = cv2.imread(img_path)
    img_rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_tensor = T.ToTensor()(img_rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_tensor)[0]
    boxes   = outputs['boxes'].cpu().numpy()
    labels  = outputs['labels'].cpu().numpy()
    scores  = outputs['scores'].cpu().numpy()
    detections = []
    for box, label, score in zip(boxes, labels, scores):
        if score < conf_thresh:
            continue
        fdi = fdi_names[label - 1]
        detections.append({
            'fdi'       : fdi,
            'confidence': round(float(score), 3),
            'bbox'      : [round(v) for v in box.tolist()],
        })
    detections.sort(key=lambda x: int(x['fdi']))
    return detections, img_rgb


def visualize_predictions(img_rgb, detections, save_path=None):
    quadrant_colors = {'1': '#FF6B6B', '2': '#4ECDC4', '3': '#45B7D1', '4': '#FFA07A'}
    fig, ax = plt.subplots(1, 1, figsize=(18, 8))
    ax.imshow(img_rgb, cmap='gray')
    for det in detections:
        fdi             = det['fdi']
        x1, y1, x2, y2 = det['bbox']
        color           = quadrant_colors.get(fdi[0], '#FFFFFF')
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-4, f"{fdi} ({det['confidence']:.2f})",
                color=color, fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.5))
    ax.set_title(f"Detected {len(detections)} teeth: {[d['fdi'] for d in detections]}", fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()


model, FDI_NAMES = load_model(f"{SAVE_DIR}/best.pt", DEVICE)

test_imgs = glob.glob(f"{DATASET_ROOT}/test/images/*")
if not test_imgs:
    test_imgs = glob.glob(f"{DATASET_ROOT}/valid/images/*")

for img_path in test_imgs[:5]:
    detections, img_rgb = predict_opg(model, img_path, FDI_NAMES, DEVICE)
    print(f"\nImage: {os.path.basename(img_path)}")
    print(f"  Detected {len(detections)} teeth: {[d['fdi'] for d in detections]}")
    for d in detections:
        print(f"    Tooth {d['fdi']}  conf={d['confidence']}  bbox={d['bbox']}")
    visualize_predictions(img_rgb, detections, save_path=f"{SAVE_DIR}/pred_{os.path.basename(img_path)}")


NameError: name 'SAVE_DIR' is not defined

---
## Cell 8 — Predict on your own uploaded image

In [ ]:
# ─────────────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

detections, img_rgb = predict_opg(model, img_path, FDI_NAMES, DEVICE, conf_thresh=0.4)

print(f"\nDetected {len(detections)} teeth:\n")
for d in detections:
    print(f"  Tooth {d['fdi']}  conf={d['confidence']}  bbox={d['bbox']}")

visualize_predictions(img_rgb, detections, save_path=f"{SAVE_DIR}/my_prediction.jpg")


---
## Cell 9 — Resume after disconnect

In [ ]:
# ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/dental_runs/resnet50_frcnn'
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

checkpoints = sorted(glob.glob(f"{SAVE_DIR}/checkpoint_epoch*.pt"))
resume_path = checkpoints[-1] if checkpoints else f"{SAVE_DIR}/best.pt"
print(f"Resuming from: {resume_path}")

ckpt        = torch.load(resume_path, map_location=DEVICE)
START_EPOCH = ckpt['epoch'] + 1
NUM_CLASSES = ckpt['num_classes']
FDI_NAMES   = ckpt['fdi_names']

model = build_model(NUM_CLASSES)
model.load_state_dict(ckpt['model_state'])
model.to(DEVICE)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
optimizer.load_state_dict(ckpt['optim_state'])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

print(f"Resuming from epoch {START_EPOCH} / 50")
print("Re-run Cell 3 first to rebuild dataloaders if /content was cleared.")
